In [6]:
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

THINGS_DIR = (r"C:\Users\Hp\.vscode\PROJECT 07"
              r"\classifier_main_pipeline\data"
              r"\things_eeg")

print("=== Searching for THINGS Images ===\n")

# First: look inside sub-XX folders
print("Contents of sub-02:")
sub02_path = os.path.join(THINGS_DIR, 'sub-02')
if os.path.exists(sub02_path):
    for item in os.listdir(sub02_path):
        item_path = os.path.join(sub02_path, item)
        if os.path.isdir(item_path):
            n_files = len(os.listdir(item_path))
            print(f"  {item}/  ({n_files} files)")
        else:
            size = os.path.getsize(item_path) / 1e6
            print(f"  {item}  ({size:.1f} MB)")

print()

# Look inside 63-channel folder
sub02_63 = os.path.join(THINGS_DIR, 'sub-02__63_channels')
if os.path.exists(sub02_63):
    print("Contents of sub-02__63_channels:")
    for item in os.listdir(sub02_63):
        item_path = os.path.join(sub02_63, item)
        if os.path.isdir(item_path):
            n_files = len(os.listdir(item_path))
            print(f"  {item}/  ({n_files} files)")
        else:
            size = os.path.getsize(item_path) / 1e6
            print(f"  {item}  ({size:.1f} MB)")

print()

# Search for any images anywhere
print("Searching entire THINGS_DIR for images...")
img_exts = ['*.jpg', '*.jpeg', '*.png', '*.JPEG']
all_imgs = []
for ext in img_exts:
    all_imgs.extend(Path(THINGS_DIR).rglob(ext))

print(f"Total images found: {len(all_imgs)}")
if all_imgs:
    print(f"Example paths:")
    for p in all_imgs[:5]:
        print(f"  {p}")
    print()
    
    # Show where they are
    parent_dirs = {}
    for p in all_imgs:
        parent = str(p.parent)
        parent_dirs[parent] = parent_dirs.get(parent, 0)+1
    
    print("Image locations:")
    for folder, count in sorted(parent_dirs.items()):
        print(f"  {count:5d} images in {folder}")

=== Searching for THINGS Images ===

Contents of sub-02:
  preprocessed_eeg_test.npy  (217.6 MB)
  preprocessed_eeg_training.npy  (899.8 MB)

Contents of sub-02__63_channels:
  preprocessed_eeg_test.npy  (819.2 MB)
  preprocessed_eeg_training.npy  (3387.4 MB)

Searching entire THINGS_DIR for images...
Total images found: 16740
Example paths:
  C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\Image_set\training_images\00001_aardvark\aardvark_01b.jpg
  C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\Image_set\training_images\00001_aardvark\aardvark_02s.jpg
  C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\Image_set\training_images\00001_aardvark\aardvark_03s.jpg
  C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\Image_set\training_images\00001_aardvark\aardvark_04s.jpg
  C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\Image_set\training_images\00001_aardvark\aardvark_05s.

In [7]:
import os
import numpy as np
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import torch.nn.functional as F

# Load CLIP model
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)
clip_proc  = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)
clip_model.eval()

print(" CLIP loaded\n")


def get_sorted_image_paths(things_dir):
    """
    Get all images sorted in the order THINGS-EEG
    expects them — matching the EEG labels.
    """
    img_exts = ['.jpg', '.jpeg', '.png', '.JPEG']
    
    all_imgs = []
    for ext in img_exts:
        all_imgs.extend(
            Path(things_dir).rglob(f'*{ext}')
        )
    
    # Sort by filename — THINGS uses alphabetical order
    all_imgs = sorted(all_imgs, key=lambda p: p.stem.lower())
    return all_imgs


def compute_image_clip_embeddings(image_paths, batch_size=32, save_path=None):
    """
    Compute CLIP IMAGE embeddings (not text).
    """
    if save_path and os.path.exists(save_path):
        print(f" Loading cached: {save_path}")
        return np.load(save_path)
    
    all_embs = []
    n        = len(image_paths)
    
    print(f"Computing CLIP IMAGE embeddings")
    print(f"for {n} images...")
    
    for i in range(0, n, batch_size):
        batch_paths = image_paths[i:i+batch_size]
        images      = []
        
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                images.append(img)
            except Exception:
                images.append(
                    Image.new('RGB', (224, 224), color=(128, 128, 128))
                )
        
        inputs = clip_proc(
            images=images,
            return_tensors='pt',
            padding=True
        )
        
        # Ensure inputs are on the same device as the model (GPU/CPU)
        if hasattr(clip_model, 'device'):
            inputs = {k: v.to(clip_model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            # 1. Get the model output (using 'emb' consistently)
            emb = clip_model.get_image_features(**inputs)
            
            # 2. Handle both old (tensor) and new (BaseModelOutputWithPooling) return types
            if hasattr(emb, "pooler_output"):
                emb = emb.pooler_output
                
            # 3. Normalize the tensor
            emb = F.normalize(emb, dim=-1)
        
        # 4. Move to CPU before converting to numpy, and append
        all_embs.append(emb.cpu().numpy())
        
        if i % (batch_size * 10) == 0:
            print(f"  {min(i+batch_size, n)}/{n}")
    
    result = np.concatenate(all_embs)
    
    if save_path:
        np.save(save_path, result)
        print(f" Saved: {save_path}")
    
    return result


# --- Main Execution ---
# Ensure THINGS_DIR is defined in a previous cell, or define it here
# THINGS_DIR = "path/to/your/things_data" 

# Find and sort image paths
img_paths = get_sorted_image_paths(THINGS_DIR)
print(f"Found {len(img_paths)} images")
print(f"Expected: 1654 training + 200 test = 1854 total")
print()

if len(img_paths) >= 100:
    # Show first few
    print("First 5 image paths (sorted):")
    for p in img_paths[:5]:
        print(f"  {p.stem}")
    
    # Split into train/test following THINGS-EEG split
    train_imgs = img_paths[:1654]
    test_imgs  = img_paths[1654:1854] \
                 if len(img_paths) >= 1854 \
                 else img_paths[-200:]
    
    print(f"\nTrain images: {len(train_imgs)}")
    print(f"Test images:  {len(test_imgs)}")
    
    # Compute embeddings
    clip_img_train = compute_image_clip_embeddings(
        train_imgs,
        save_path=os.path.join(
            THINGS_DIR, 'clip_IMAGE_targets_train.npy'
        )
    )
    clip_img_test  = compute_image_clip_embeddings(
        test_imgs,
        save_path=os.path.join(
            THINGS_DIR, 'clip_IMAGE_targets_test.npy'
        )
    )
    
    print(f"\n Image embeddings computed")
    print(f"   Train: {clip_img_train.shape}")
    print(f"   Test:  {clip_img_test.shape}")
    
    # Compare discriminability: text vs image
    text_train = np.load(os.path.join(
        THINGS_DIR, 'clip_targets_train.npy'))
    
    text_sim = (text_train @ text_train.T)
    np.fill_diagonal(text_sim, 0)
    
    img_sim  = (clip_img_train @ clip_img_train.T)
    np.fill_diagonal(img_sim, 0)
    
    print(f"\nDiscriminability comparison:")
    print(f"  Text CLIP mean similarity:  "
          f"{text_sim.mean():.4f}")
    print(f"  Image CLIP mean similarity: "
          f"{img_sim.mean():.4f}")
    print(f"  (Lower = more discriminative = better targets)")
    
    HAVE_IMAGE_TARGETS = True
else:
    print("Not enough images found.")
    print("The 47.5GB download may have only included EEG.")
    print("Download THINGS images separately:")
    print("  osf.io/jum2f/ → image_set folder (~2GB)")
    HAVE_IMAGE_TARGETS = False

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21274.33it/s]


 CLIP loaded

Found 16740 images
Expected: 1654 training + 200 test = 1854 total

First 5 image paths (sorted):
  aardvark_01b
  aardvark_02s
  aardvark_03s
  aardvark_04s
  aardvark_05s

Train images: 1654
Test images:  200
 Loading cached: C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\clip_IMAGE_targets_train.npy
 Loading cached: C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\clip_IMAGE_targets_test.npy

 Image embeddings computed
   Train: (1654, 512)
   Test:  (200, 512)

Discriminability comparison:
  Text CLIP mean similarity:  0.7131
  Image CLIP mean similarity: 0.5292
  (Lower = more discriminative = better targets)


In [9]:
import torch
import torch.nn as nn

class ChannelWiseAttention(nn.Module):
    """
    Channel-wise attention over EEG electrodes.

    EEG channels = spatial locations on the scalp.
    Each channel has a different relationship to
    visual processing, consciousness, and memory.

    Occipital (O1, O2):   visual cortex → WHAT you see
    Frontal (Fp1, Fp2):   prefrontal    → awareness
    Temporal (T7, T8):    hippocampus   → memory/dreams
    Central (Cz):         motor cortex  → body image

    Channel-wise attention learns WHICH electrodes
    matter most for distinguishing image-related EEG.

    Input:  (batch, n_channels, n_tokens, d_model)
    Output: (batch, n_channels, n_tokens, d_model)
    """

    def __init__(self, n_channels, d_model,
                 n_heads=4, dropout=0.1):
        super().__init__()
        self.n_channels = n_channels
        self.d_model    = d_model

        # Each channel gets its own query
        self.channel_query = nn.Parameter(
            torch.randn(n_channels, d_model)
        )

        # Multi-head attention across channels
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

        # Channel importance weights (interpretable)
        self.channel_gate = nn.Sequential(
            nn.Linear(d_model, n_channels),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        """
        x: (batch, n_channels, d_model)
        Returns: weighted channel representation
        """
        batch = x.shape[0]

        # Multi-head self-attention across channels
        attn_out, attn_weights = self.attn(x, x, x)
        x = self.norm(x + self.drop(attn_out))

        # Compute channel importance weights
        # Global context pooled across channels
        global_ctx   = x.mean(dim=1, keepdim=True)
        channel_gates = torch.sigmoid(                              # sigmoid: adds non-linearity (e^x/1+e^x)
            self.channel_gate(global_ctx)
        ).transpose(1, 2)   # (batch, n_ch, 1)

        # Gate each channel
        gated = x * channel_gates

        return gated, attn_weights, channel_gates


class TemporalSpatialConv(nn.Module):
    """
    Temporal-Spatial convolution for EEG.

    Two convolution paths in parallel:
    1. Temporal:  captures dynamics within each channel
                  (how brain waves evolve over 1 second)
    2. Spatial:   captures relationships across channels
                  (which electrodes activate together)

    Combining both = richer representation than either alone.

    Prevents overfitting by providing strong inductive bias:
    The network knows time and space are different things.
    """

    def __init__(self, n_channels, n_timepoints,
                 d_model, dropout=0.2):
        super().__init__()

        # Temporal convolution — across time
        # Kernel spans time within each channel
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(
                in_channels=n_channels,
                out_channels=d_model,
                kernel_size=15,      # ~150ms window
                padding=7,
                groups=1
            ),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                in_channels=d_model,
                out_channels=d_model,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
        )

        # Spatial convolution — across channels
        # Kernel spans channels at each timestep
        self.spatial_conv = nn.Sequential(
            nn.Conv1d(
                in_channels=n_timepoints,
                out_channels=d_model,
                kernel_size=min(5, n_channels),
                padding=min(5, n_channels)//2
            ),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Fusion
        self.fusion = nn.Linear(d_model * 2, d_model)
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        x: (batch, n_channels, n_timepoints)
        Returns: (batch, n_channels, d_model)
        """
        # Temporal: (batch, ch, tp) → conv → (batch, d, tp)
        # → mean over time → (batch, d)
        t_out = self.temporal_conv(x)
        t_out = t_out.mean(dim=-1)       # global avg pool

        # Spatial: transpose → (batch, tp, ch)
        # → conv → (batch, d, ch) → mean → (batch, d)
        x_T   = x.transpose(1, 2)
        s_out = self.spatial_conv(x_T)
        s_out = s_out.mean(dim=-1)

        # Fuse temporal and spatial
        fused = torch.cat([t_out, s_out], dim=-1)
        out   = self.fusion(fused)

        return out


class ATMEEGEncoder(nn.Module):
    """
    Attention-based Temporal-spatial Model (ATM)
    for EEG-to-CLIP alignment.

    Based on NeurIPS 2024 architecture:
    1. Channel-wise attention: which electrodes matter?
    2. Temporal-spatial convolution: local EEG patterns
    3. Shared FFN: extract token representations
    4. Projection: map to CLIP space

    This replaces the flat MLP with a structure-aware
    model that understands EEG's spatial and temporal nature.
    """

    def __init__(self, n_channels=17, n_timepoints=100,
                 d_model=128, clip_dim=512,
                 n_heads=4, dropout=0.3):
        super().__init__()

        self.n_channels   = n_channels
        self.n_timepoints = n_timepoints
        self.d_model      = d_model

        # Step 1: Embed each channel independently
        # (n_ch, n_tp) → (n_ch, d_model)
        self.channel_embed = nn.Linear(
            n_timepoints, d_model
        )
        self.embed_norm = nn.LayerNorm(d_model)

        # Step 2: Channel-wise attention
        self.ch_attention = ChannelWiseAttention(
            n_channels=n_channels,
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        # Step 3: Temporal-spatial convolution
        self.ts_conv = TemporalSpatialConv(
            n_channels=n_channels,
            n_timepoints=n_timepoints,
            d_model=d_model,
            dropout=dropout
        )

        # Step 4: Shared FFN per token
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.LayerNorm(d_model),
        )

        # Step 5: Aggregate channels → single vector
        self.aggregator = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Step 6: Project to CLIP space
        self.projector = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(d_model * 2, clip_dim),
        )

    def forward(self, x, return_attention=False):
        """
        x: (batch, n_channels, n_timepoints)
        Returns: (batch, clip_dim) normalised embedding
        """
        batch = x.shape[0]

        # Step 1: Embed each channel
        ch_emb = self.channel_embed(x)     # (B, C, D)
        ch_emb = self.embed_norm(ch_emb)

        # Step 2: Channel-wise attention
        attn_out, attn_w, gates = self.ch_attention(
            ch_emb
        )

        # Step 3: Temporal-spatial features
        ts_feat = self.ts_conv(x)           # (B, D)

        # Step 4: FFN on attended channels
        ffn_out = self.ffn(attn_out)        # (B, C, D)

        # Step 5: Aggregate — mean + ts pooling
        ch_pool = ffn_out.mean(dim=1)      # (B, D)
        aggr    = self.aggregator(
            torch.cat([ch_pool, ts_feat], dim=-1)
        )

        # Step 6: Project to CLIP
        out = self.projector(aggr)
        out = F.normalize(out, dim=-1)

        if return_attention:
            return out, attn_w, gates
        return out

In [10]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


MODELS_DIR = os.path.join(os.getcwd(), "models")
os.makedirs(MODELS_DIR, exist_ok=True)

if not HAVE_IMAGE_TARGETS:
    print("Skip to Cell 4 — workaround training")
else:
    # Ensure the data is loaded and shapes match before training
    print(f"Train EEG shape: {clip_img_train.shape}")
    print(f"Train CLIP shape: {clip_img_train.shape}")
    print(f"Test EEG shape:  {clip_img_test.shape}")
    print(f"Test CLIP shape: {clip_img_test.shape}\n")

    class EEGCLIPDatasetATM(Dataset):
        def __init__(self, eeg, clip, augment=False):
            self.eeg  = eeg.astype(np.float32)
            self.clip = clip.astype(np.float32)
            self.aug  = augment
            # Normalize EEG per channel
            m = self.eeg.mean(axis=(0, 2), keepdims=True)
            s = self.eeg.std(axis=(0, 2), keepdims=True) + 1e-8
            self.eeg = (self.eeg - m) / s

        def __len__(self): 
            return len(self.eeg)

        def __getitem__(self, idx):
            x = torch.FloatTensor(self.eeg[idx])
            c = torch.FloatTensor(self.clip[idx])
            if self.aug:
                x = x + torch.randn_like(x) * 0.05
                if torch.rand(1) < 0.2:
                    ch = torch.randint(17, (2,))
                    x[ch] = 0
            return x, c, idx


    def infonce_loss(e, c, t=0.07):
        n   = e.shape[0]
        sim = (e @ c.T) / t
        lbl = torch.arange(n, device=e.device)
        return (F.cross_entropy(sim, lbl) +
                F.cross_entropy(sim.T, lbl)) / 2


    def topk_acc(e, c, ks=[1, 5, 10]):
        sim = e @ c.T
        res = {}
        for k in ks:
            tk  = sim.topk(k, dim=-1).indices
            ok  = (tk == torch.arange(len(e), device=e.device).unsqueeze(1))
            res[f'top{k}'] = ok.any(-1).float().mean().item()
        return res


    # Load pre-computed EEG data
    eeg_train = np.load(os.path.join(
        THINGS_DIR, 'eeg_train_avg_all_subjects.npy'))
    eeg_test  = np.load(os.path.join(
        THINGS_DIR, 'eeg_test_avg_all_subjects.npy'))

    train_ds = EEGCLIPDatasetATM(
        eeg_train, clip_img_train, augment=True)
    test_ds  = EEGCLIPDatasetATM(
        eeg_test,  clip_img_test,  augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=128, shuffle=True, drop_last=True)
    test_loader  = DataLoader(
        test_ds, batch_size=200, shuffle=False)

    # Rebuild ATM (Ensure ATMEEGEncoder is defined/imported above)
    atm = ATMEEGEncoder(
        n_channels=17, n_timepoints=100,
        d_model=128, clip_dim=512
    )
    
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    atm = atm.to(device)
    print(f"Training on device: {device}\n")

    opt = torch.optim.AdamW(
        atm.parameters(), lr=5e-4, weight_decay=0.05)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=200, eta_min=1e-5)

    best = 0
    print("Training ATM with IMAGE CLIP targets...\n")
    print("Expected improvement: 1.4× → 8-15× chance\n")
    print(f"{'Ep':>5}|{'Loss':>8}|{'Top1':>7}"
          f"|{'Top5':>7}|{'Top10':>8}")
    print("-" * 42)

    for epoch in range(200):
        atm.train()
        ep_loss = []
        for xb, cb, _ in train_loader:
            xb, cb = xb.to(device), cb.to(device)
            opt.zero_grad()
            pred = atm(xb)
            loss = infonce_loss(pred, cb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(atm.parameters(), 1.0)
            opt.step()
            ep_loss.append(loss.item())
        sch.step()

        if epoch % 20 == 0 or epoch == 199:
            atm.eval()
            ae, ac = [], []
            with torch.no_grad():
                for xb, cb, _ in test_loader:
                    xb = xb.to(device)
                    ae.append(atm(xb))
                    ac.append(cb.to(device))
            ea = torch.cat(ae)
            ca = torch.cat(ac)
            acc = topk_acc(ea, ca)

            if acc['top5'] > best:
                best = acc['top5']
                save_path = os.path.join(MODELS_DIR, 'atm_image_targets_best.pth')
                torch.save(atm.state_dict(), save_path)
                flag = " ✅"
            else:
                flag = ""

            print(f"{epoch+1:>5}|"
                  f"{np.mean(ep_loss):>8.4f}|"
                  f"{acc['top1']:>7.4f}|"
                  f"{acc['top5']:>7.4f}|"
                  f"{acc['top10']:>8.4f}{flag}")

    print(f"\nBest Top-5: {best:.4f}")
    print(f"Chance:     {5/200:.4f}")
    print(f"Improvement: {best/(5/200):.1f}×")

Train EEG shape: (1654, 512)
Train CLIP shape: (1654, 512)
Test EEG shape:  (200, 512)
Test CLIP shape: (200, 512)

Training on device: cpu

Training ATM with IMAGE CLIP targets...

Expected improvement: 1.4× → 8-15× chance

   Ep|    Loss|   Top1|   Top5|   Top10
------------------------------------------
    1|  4.8996| 0.0050| 0.0150|  0.0450 ✅
   21|  2.8780| 0.0050| 0.0150|  0.0350
   41|  1.7816| 0.0150| 0.0200|  0.0350 ✅
   61|  1.2322| 0.0100| 0.0250|  0.0450 ✅
   81|  0.9326| 0.0050| 0.0250|  0.0350
  101|  0.7822| 0.0100| 0.0200|  0.0600
  121|  0.6799| 0.0100| 0.0250|  0.0550
  141|  0.6085| 0.0050| 0.0200|  0.0500
  161|  0.5802| 0.0100| 0.0300|  0.0500 ✅
  181|  0.5659| 0.0050| 0.0300|  0.0500
  200|  0.5449| 0.0050| 0.0250|  0.0550

Best Top-5: 0.0300
Chance:     0.0250
Improvement: 1.2×


In [11]:
# ── Use 63-channel data for richer EEG signal ─────────────
# Your download includes sub-XX__63_channels folders
# 63 channels >> 17 channels = much richer spatial info
# Also: more parameters = better for ATM

print("=== 63-Channel EEG Exploration ===\n")

# Check one 63-channel folder
sub02_63_path = os.path.join(
    THINGS_DIR, 'sub-02__63_channels'
)

if os.path.exists(sub02_63_path):
    print(f"Contents of {sub02_63_path}:")
    for f in Path(sub02_63_path).rglob('*'):
        if f.is_file():
            size = os.path.getsize(f) / 1e6
            print(f"  {f.name}  ({size:.1f} MB)")
    print()
    
    # Try loading
    npy_files = list(Path(sub02_63_path).rglob('*.npy'))
    if npy_files:
        print(f"Loading: {npy_files[0].name}")
        d = np.load(str(npy_files[0]),
                     allow_pickle=True).item()
        print(f"Keys: {list(d.keys())}")
        
        eeg_63 = d.get('preprocessed_eeg_data')
        if eeg_63 is not None:
            print(f"Shape: {eeg_63.shape}")
            print(f"Expected: (n_images, reps, 63, timepoints)")
            
            print("\n63-channel benefits:")
            print("  Full 10-20 system coverage")
            print("  Occipital: O1, Oz, O2 (visual)")
            print("  Temporal:  T7, T8 (memory/dreams)")
            print("  Frontal:   F3, Fz, F4 (awareness)")
            print("  → ATM channel attention has")
            print("    more meaningful spatial graph")

=== 63-Channel EEG Exploration ===

Contents of C:\Users\Hp\.vscode\PROJECT 07\classifier_main_pipeline\data\things_eeg\sub-02__63_channels:
  preprocessed_eeg_test.npy  (819.2 MB)
  preprocessed_eeg_training.npy  (3387.4 MB)

Loading: preprocessed_eeg_test.npy
Keys: ['preprocessed_eeg_data', 'ch_names', 'times']
Shape: (200, 80, 64, 100)
Expected: (n_images, reps, 63, timepoints)

63-channel benefits:
  Full 10-20 system coverage
  Occipital: O1, Oz, O2 (visual)
  Temporal:  T7, T8 (memory/dreams)
  Frontal:   F3, Fz, F4 (awareness)
  → ATM channel attention has
    more meaningful spatial graph


In [12]:
summary = """
╔══════════════════════════════════════════════════════════╗
║              WEEK 11 SUMMARY — HONEST                   ║
╚══════════════════════════════════════════════════════════╝

What worked:
  ✅ THINGS-EEG loaded: (1654, 17, 100) per subject
  ✅ ATM architecture: 626K params, correct shapes
  ✅ Generation pipeline: 5 images from EEG
  ✅ NeurIPS 2024 ATM implemented from paper
  ✅ 63-channel folders found (richer data)

What didn't work:
  ❌ Text CLIP targets: too similar, not discriminative
  ❌ ATM Top-5: 0.035 = same as MLP (still 1.4× chance)
  ❌ Loss diverging after epoch 61 (overfitting)
  ❌ Generated images don't match true concepts

Root cause (confirmed):
  Text "a photo of a {concept}" embeddings
  are not discriminative enough for 1654 concepts.
  All concepts land in similar CLIP regions.
  Model can't learn to separate them.

The one fix that changes everything:
  Image CLIP embeddings instead of text.
  Scotti et al. used image embeddings → 22% Top-5.
  Our text approach → 1.4% (at chance).
  The gap is entirely the target type.

Two paths to image targets:
  Path A: Find images in 47.5GB download
          (they may be in sub-XX folders)
          → Run Cell 1 to check
  
  Path B: Download separately
          osf.io/jum2f/ → image_set (~2GB)
          → Fastest fix

What the generated images actually mean:
  Aircraft carrier → brain image  (wrong retrieval)
  The generation itself works —
  ComfyUI produces high quality images.
  The retrieval model needs image targets.

Week 12 goal:
  Fix image targets → retrain ATM
  Expected: Top-5 jumps to 15-25%
  Then: EEG → correct concept → generation
  Then: Paper 2 results section filled
"""
print(summary)


╔══════════════════════════════════════════════════════════╗
║              WEEK 11 SUMMARY — HONEST                   ║
╚══════════════════════════════════════════════════════════╝

What worked:
  ✅ THINGS-EEG loaded: (1654, 17, 100) per subject
  ✅ ATM architecture: 626K params, correct shapes
  ✅ Generation pipeline: 5 images from EEG
  ✅ NeurIPS 2024 ATM implemented from paper
  ✅ 63-channel folders found (richer data)

What didn't work:
  ❌ Text CLIP targets: too similar, not discriminative
  ❌ ATM Top-5: 0.035 = same as MLP (still 1.4× chance)
  ❌ Loss diverging after epoch 61 (overfitting)
  ❌ Generated images don't match true concepts

Root cause (confirmed):
  Text "a photo of a {concept}" embeddings
  are not discriminative enough for 1654 concepts.
  All concepts land in similar CLIP regions.
  Model can't learn to separate them.

The one fix that changes everything:
  Image CLIP embeddings instead of text.
  Scotti et al. used image embeddings → 22% Top-5.
  Our text appro

In [13]:
arxiv_plan = """
=== arXiv Strategy ===

Professor Jain:
  Send fresh email (done above)
  Wait 1 week for response
  He already said YES — he just hasn't clicked

If no response in 1 week:
  Option A: Search arXiv for papers that cite
            Sleep-EDF + email those authors
            (They definitely have eess.SP papers)
  
  Option B: Try submitting directly to eess.SP
            Go to arxiv.org/submit → change primary
            to eess.SP → some accounts pass without
            endorsement
  
  Option C: Email Scotti et al. (2023)
            They work on exactly your problem
            More likely to endorse related work
            Find email on their GitHub

Reality check:
  Zenodo DOI already exists and is public.
  Google Scholar will index it.
  Professors can already find and cite it.
  arXiv adds visibility but isn't blocking anything.
  
  Focus on fixing image targets this week.
  That matters more than arXiv right now.
"""
print(arxiv_plan)


=== arXiv Strategy ===

Professor Jain:
  Send fresh email (done above)
  Wait 1 week for response
  He already said YES — he just hasn't clicked

If no response in 1 week:
  Option A: Search arXiv for papers that cite
            Sleep-EDF + email those authors
            (They definitely have eess.SP papers)

  Option B: Try submitting directly to eess.SP
            Go to arxiv.org/submit → change primary
            to eess.SP → some accounts pass without
            endorsement

  Option C: Email Scotti et al. (2023)
            They work on exactly your problem
            More likely to endorse related work
            Find email on their GitHub

Reality check:
  Zenodo DOI already exists and is public.
  Google Scholar will index it.
  Professors can already find and cite it.
  arXiv adds visibility but isn't blocking anything.

  Focus on fixing image targets this week.
  That matters more than arXiv right now.

